In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, FuncFormatter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm
import os

from model import *
from data import generate_gmm_data, generate_random_data
from train_utils import train, train_epoch

from plot import *
from utils import *



In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: X
d_identity = lambda X: np.ones(X.shape)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)

leaky_softplus = lambda X, slope=0.01: slope * X + (1 - slope) * (softplus(X) - np.log(2))
d_leaky_softplus = lambda X, slope=0.01: slope + (1 - slope) * (d_softplus(X))


smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'Leaky-Softplus': [leaky_softplus, d_leaky_softplus], 
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs],
               'Identity': [identity, d_identity]}

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

optimizer_map = {'SGD': optim.SGD,
                'Adam': optim.Adam}

# Model Training

In [ ]:
def get_data(d: int, 
             K: int, 
             N_k: int, 
             nu: float = 1.0,
             normalize_: bool = False, 
             orthogonalize_: bool = False, 
             batch_size: int = None):
    
    
    N = K * N_k
    
    train_set, train_loader, grad_loader = generate_gmm_data(d, 
                                                K, 
                                                N_k, 
                                                nu=nu,
                                                whiten_ = whiten_, 
                                                normalize_ = normalize_, 
                                                batch_size = batch_size)
    val_loader = train_loader
    
    
    X = train_set.samples.T
    Y = train_set.targets.T
    
    return train_loader, val_loader, grad_loader, X, Y

In [ ]:
def get_init_grads_and_outs(model, dataloader, optimizer, criterion):
    
    results = train_epoch(model, 
                        dataloader, 
                        optimizer, 
                        criterion, 
                        device,
                        update=False, 
                        return_grads=True, 
                        return_outs=True)
    
    return results['epoch_grads'], results['epoch_outs'] 
    

In [ ]:
def train_model(num_layers: int,
                data_dim: int, 
                hidden_dim: int, 
                num_classes: int, 
                act_fn_str: str, 
                train_loader, 
                grad_loader, 
                val_loader, 
                epochs, 
                lr, 
                freq,
                loss_fn, 
                optimizer_type='SGD', 
                freeze_last_layer=True, 
                diff_init_last_layer=False,
                residual=False, 
                batchnorm=False, 
                init_method='orthogonal',
                init_scale=1):
    
    
    # Train model
    model = MLP(num_layers, 
                data_dim, 
                hidden_dim, 
                num_classes, 
                act_fn_str, 
                freeze_last_layer=freeze_last_layer,
                diff_init_last_layer=diff_init_last_layer,
                residual=residual, 
                batchnorm=batchnorm, 
                init_method=init_method, 
                init_scale=init_scale)

    model = model.to(device)

    # Training
    criterion = loss_map[loss_fn]
    optimizer = optimizer_map[optimizer_type](model.parameters(), lr=lr, weight_decay=0)
    scheduler = None
    
    init_grads, init_outs = get_init_grads_and_outs(model, grad_loader, optimizer, criterion)
    results = train(model, 
                    criterion, 
                    optimizer, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=scheduler, 
                    return_weights=True,
                    return_grads=False,
                    return_outs=False)


    all_weights = results['all_weights']
    train_losses = results['train_losses']
    
    return init_grads, init_outs, all_weights, train_losses

# Weight decompositions during training

In [ ]:
# Data setup
d = 32 # Ambient dim
m = d
K = 4 # Number of classes
N_k = 500
N = N_k * K
nu = np.sqrt(3)

# Model setup
L = 4
act_fn_strs = ['GELU', 'SiLU', 'ELU', 'LeakyReLU', 'ReLU', 'RReLU']
#act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
freeze_last_layer = False
diff_init_last_layer = False
init_method = 'orthogonal'
init_scale = 1

whiten_ = True
normalize_ = False
orthogonalize_ = False
batch_size = None

# Training setup
epochs = 100
lrs = [3e-3, 3e-3, 1e-3, 1e-3, 1e-3, 1e-3] 
#lrs = [5e-4, 5e-4, 1e-4, 3e-4, 3e-4, 3e-4]
freq = 1

optimizer_type = 'SGD'

#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

savefig_dir = './figs/main_fig/all' if whiten_ else './figs/main_fig/all_unwhitened'
if not os.path.exists(savefig_dir):
    os.makedirs(savefig_dir)

    
save = True


In [ ]:
trials = 10

train_losses = np.zeros((trials, len(act_fn_strs), epochs))

U_top_errors = np.zeros((trials, len(act_fn_strs), epochs, L - 1))
U_mid_errors = np.zeros((trials, len(act_fn_strs), epochs, L - 1))
U_bot_errors = np.zeros((trials, len(act_fn_strs), epochs, L - 1))

V_top_errors = np.zeros((trials, len(act_fn_strs), epochs, L - 1))
V_mid_errors = np.zeros((trials, len(act_fn_strs), epochs, L - 1))
V_bot_errors = np.zeros((trials, len(act_fn_strs), epochs, L - 1))

svals = np.zeros((trials, len(act_fn_strs), epochs, L - 1, d))

for (k, act_fn_str) in enumerate(act_fn_strs):
    lr = lrs[k]
    print("===== ACTIVATION FUNCTION: ", act_fn_str, " =====")
    for i in range(trials):
        # Create data
        train_loader, val_loader, grad_loader, X, Y = get_data(d, K, N_k, nu, whiten_, normalize_, batch_size=batch_size)
        
        print("=== TRIAL " + str(i) + " ===")
        # Train model
        init_grads, init_outs, all_weights, train_losses_trial = train_model(L, d, m, K, act_fn_str, 
                                                                 train_loader, 
                                                                 grad_loader, 
                                                                 val_loader, 
                                                                 epochs, 
                                                                 lr, 
                                                                 freq, 
                                                                 loss_fn, 
                                                                 optimizer_type=optimizer_type,
                                                                 freeze_last_layer=freeze_last_layer,
                                                                 diff_init_last_layer=diff_init_last_layer,
                                                                 residual=residual,
                                                                 batchnorm=batchnorm,
                                                                 init_method=init_method,
                                                                 init_scale=init_scale)


            
        # Plot train loss
        plot_train_losses(train_losses_trial, act_fn_str, d, K, None, savefig=False)
        
        train_losses[i, k, :] = train_losses_trial


        # Calculate norms of decompositions
        errors = save_weight_svd_errors(all_weights, 
                                        init_grads[0],
                                        K, 
                                        init_scale)
        
        U_top_errors[i, k, ...] = errors['U_top_errors']
        U_mid_errors[i, k, ...] = errors['U_mid_errors']
        U_bot_errors[i, k, ...] = errors['U_bot_errors']
        
        V_top_errors[i, k, ...] = errors['V_top_errors']
        V_mid_errors[i, k, ...] = errors['V_mid_errors']
        V_bot_errors[i, k, ...] = errors['V_bot_errors']
        
        svals[i, k, ...] = errors['svals']
    
    
# Save results
save_fname = os.path.join(savefig_dir, 'all_acts_losses_errors_dists.npz') if whiten_ else os.path.join(savefig_dir, 'all_acts_losses_errors_dists_unwhitened.npz')

if save:
    np.savez(save_fname, 
             train_losses=train_losses, 
             U_top_errors=U_top_errors, 
             U_mid_errors=U_mid_errors, 
             U_bot_errors=U_bot_errors,
             V_top_errors=V_top_errors, 
             V_mid_errors=V_mid_errors, 
             V_bot_errors=V_bot_errors, 
             svals=svals)

In [ ]:
mean_U_top_errors = np.mean(U_top_errors, axis=0)
std_U_top_errors = np.std(U_top_errors, axis=0)

mean_U_mid_errors = np.mean(U_mid_errors, axis=0)
std_U_mid_errors = np.std(U_mid_errors, axis=0)

mean_U_bot_errors = np.mean(U_bot_errors, axis=0)
std_U_bot_errors = np.std(U_bot_errors, axis=0)


mean_V_top_errors = np.mean(V_top_errors, axis=0)
std_V_top_errors = np.std(V_top_errors, axis=0)

mean_V_mid_errors = np.mean(V_mid_errors, axis=0)
std_V_mid_errors = np.std(V_mid_errors, axis=0)

mean_V_bot_errors = np.mean(V_bot_errors, axis=0)
std_V_bot_errors = np.std(V_bot_errors, axis=0)

mean_svals = np.mean(svals, axis=0)

# Smooth vs. Nonsmooth Fig

In [ ]:
def log_errorbar(y, std):
    y_low = y * np.exp(-std / y)
    y_high = y * np.exp(std / y)
    return np.vstack((y - y_low, y_high - y))

def plot_series(
    ax,
    series,
    layer=0,
    color="viridis",
    crossing_lines=False,
    x_points=None,
    y_points=None,
    zoom=0.8,
    elev=30,
    azim=-50,
    roll=0,
    alpha=1.0,
    linewidth=1.0,
):
    n_x_indices, n_y_indices = series.shape[0], series.shape[-1]
    x_indices = np.arange(n_x_indices) + 1
    y_indices = np.arange(n_y_indices) + 1

    if x_points is None:
        x_points = x_indices
    if y_points is None:
        y_points = y_indices

    spectrum_verts = []

    for idx in x_indices:
        spectrum_verts.append(
            [
                (0, np.min(series) - 0.1),
                *zip(y_points, series[idx - 1, layer, :]),
                (y_points[-1], np.min(series) - 0.1),
            ]
        )
    
    
    

    stride=5
    
    spectrum_poly = PolyCollection(spectrum_verts[::stride]) #, facecolors='none')
    spectrum_poly.set_linewidth(linewidth)
    spectrum_poly.set_alpha(alpha)
    spectrum_poly.set_facecolor(
        plt.colormaps[color](np.linspace(0.3, 0.8, len(spectrum_verts[::stride])))  # type: ignore
    )
    spectrum_poly.set_edgecolor("black")
    
    
#     colors = plt.colormaps[color](np.linspace(0.5, 1.0 len(spectrum_verts)))
#     spectrum_poly.set_edgecolor(colors)
#     spectrum_poly.set_linewidth(0.8)
#     spectrum_poly.set_alpha(0.9)
    
    #spectrum_poly.set_antialiased(False)

    ax.set_box_aspect(aspect=None, zoom=zoom)

    ax.add_collection3d(spectrum_poly, zs=x_indices[::stride], zdir="y")

    if crossing_lines:
        path_verts = []

        for idx in y_indices:
            path_verts.append([*zip(x_points, series[:, layer, idx - 1])])
        path_line = LineCollection(path_verts)
        path_line.set_linewidth(linewidth)
        path_line.set_edgecolor("black")
        ax.add_collection3d(path_line, zs=y_indices, zdir="x")

    ax.set_xlim(y_points[0], y_points[-1])
    ax.set_ylim(x_points[0], x_points[-1])
    ax.set_zlim(np.min(series) - 0.1, np.max(series) + 0.1)

    ax.view_init(elev, azim, roll)
    ax.xaxis.set_major_locator(MultipleLocator(8)) #MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MultipleLocator(25)) #MaxNLocator(integer=True))
    
    ax.xaxis.labelpad=8
    ax.yaxis.labelpad=8
    

def plot_weight_svd_errors(mean_errors_dict: Dict[str, np.ndarray], 
                           std_errors_dict: Dict[str, np.ndarray],
                           act_str: str, 
                           d: int, 
                           K: int, 
                           save_dir: str,
                           savefig: bool = False, 
                           whitened: bool = True):
    svals = mean_errors_dict['svals']

    mean_U_top_errors = mean_errors_dict["U_top_errors"]
    std_U_top_errors = std_errors_dict["U_top_errors"]
    
    mean_U_mid_errors = mean_errors_dict["U_mid_errors"]
    std_U_mid_errors = std_errors_dict["U_mid_errors"]
    
    mean_U_bot_errors = mean_errors_dict["U_bot_errors"]
    std_U_bot_errors = std_errors_dict["U_bot_errors"]
    
    
    mean_V_top_errors = mean_errors_dict["V_top_errors"]
    std_V_top_errors = std_errors_dict["V_top_errors"]
    
    mean_V_mid_errors = mean_errors_dict["V_mid_errors"]
    std_V_mid_errors = std_errors_dict["V_mid_errors"]
    
    mean_V_bot_errors = mean_errors_dict["V_bot_errors"]
    std_V_bot_errors = std_errors_dict["V_bot_errors"]
    

    epochs, L = mean_U_top_errors.shape
    
    colors = {
        "top": "tab:blue",
        "mid": "tab:orange",
        "bot": "tab:green"
    }

    linestyles = {
        "top": "-",
        "mid": "--",
        "bot": ":",
    }

    markers = {
        "L": "o",
        "R": "s"
    }

    lw_L, lw_R = 2.6, 1.9
    alpha_L, alpha_R = 0.9, 0.7
    markevery_L = (9, 10)
    markevery_R = (4, 10)
    x = np.arange(epochs) + 1


    for l in range(L):
        fig = plt.figure(figsize=(10, 5), constrained_layout=True)
        
        gs = fig.add_gridspec(
            nrows=1,
            ncols=2,
            width_ratios=[1.33, 1.0]  # ← give left plot more room
        )
        
        # Singular values
        ax1 = fig.add_subplot(gs[0, 0], projection='3d') #fig.add_subplot(131, projection='3d')
        plot_series(ax1, svals, layer=l, crossing_lines=True, linewidth=0.5, alpha=1.0, zoom=1.1)
        
        ax1.set_xlabel(r'SV Index', fontsize=20)
        ax1.set_ylabel(r'Train Epoch', fontsize=20)
#         ax1.set_zlabel(r'$\sigma_i(t)$', fontsize=16)

        ax1.set_title('Singular Values', fontsize=20)
        ax1.xaxis.set_tick_params(labelsize=14)
        ax1.yaxis.set_tick_params(labelsize=14)
        ax1.zaxis.set_tick_params(labelsize=14)
        
        
        # Singular subspaces
        # ---------- Singular subspaces ----------
        ax2 = fig.add_subplot(gs[0, 1])

        # ---- TOP ----
        ax2.errorbar(
            x,
            mean_U_top_errors[:, l],
            yerr=log_errorbar(mean_U_top_errors[:, l], std_U_top_errors[:, l]),
            label='Top L',
            color=colors["top"],
            linestyle='none', #linestyles["top"],
            marker=markers["L"],
            #mfc='none',
            markevery=markevery_L,
            errorevery=markevery_L,
            linewidth=lw_L,
            alpha=alpha_L,
            elinewidth=2, 
            capsize=3,
            markersize=8,
            zorder=3
        )
     
        ax2.errorbar(
            x,
            mean_V_top_errors[:, l],
            yerr=log_errorbar(mean_V_top_errors[:, l], std_V_top_errors[:, l]),
            label='Top R',
            color=colors["top"],
            linestyle='none', #linestyles["top"],
            marker=markers["R"],
            #mfc='none',
            markevery=markevery_R,
            errorevery=markevery_R,
            linewidth=lw_L,
            alpha=alpha_L,
            elinewidth=2, 
            capsize=3,
            markersize=8,
            zorder=3
        )

        # ---- MID ----
        ax2.errorbar(
            x,
            mean_U_mid_errors[:, l],
            yerr=log_errorbar(mean_U_mid_errors[:, l], std_U_mid_errors[:, l]),
            label='Mid. L',
            color=colors["mid"],
            linestyle='none', #linestyles["mid"],
            marker=markers["L"],
            markevery=markevery_L,
            errorevery=markevery_L,
            linewidth=lw_L,
            alpha=alpha_L,
            elinewidth=2,
            capsize=3,
            markersize=10
        )
    
        ax2.errorbar(
            x,
            mean_V_mid_errors[:, l],
            yerr=log_errorbar(mean_V_mid_errors[:, l], std_V_mid_errors[:, l]),
            label='Mid. R',
            color=colors["mid"],
            linestyle='none', #linestyles["mid"],
            marker=markers["R"],
            markevery=markevery_R,
            errorevery=markevery_R,
            linewidth=lw_L,
            alpha=alpha_L,
            elinewidth=2,
            capsize=3,
            markersize=10
        )

        # ---- BOTTOM ----
        ax2.errorbar(
            x,
            mean_U_bot_errors[:, l],
            yerr=log_errorbar(mean_U_bot_errors[:, l], std_U_bot_errors[:, l]),
            label='Bot. L',
            color=colors["bot"],
            linestyle='none', #linestyles["bot"],
            marker=markers["L"],
            #mfc='none',
            markevery=markevery_L,
            errorevery=markevery_L,
            linewidth=lw_L,
            alpha=alpha_L,
            elinewidth=2,
            capsize=3,
            markersize=12
        )

        ax2.errorbar(
            x,
            mean_V_bot_errors[:, l],
            yerr=log_errorbar(mean_V_bot_errors[:, l], std_V_bot_errors[:, l]),
            label='Bot. R',
            color=colors["bot"],
            linestyle='none', #linestyles["bot"],
            marker=markers["R"],
            #mfc='none',
            markevery=markevery_R,
            errorevery=markevery_R,
            linewidth=lw_L,
            alpha=alpha_L,
            elinewidth=2,
            capsize=3,
            markersize=12
        )
        


        #ax2.set_title(r'Singular Subspace Distance from $\mathcal{S}_l$', fontsize=20, y=1.125)
        ax2.set_title(r'Singular Subspace Change', fontsize=20, y=1.2)
        ax2.set_ylabel('Squared Distance', fontsize=20)
        ax2.tick_params(axis='x', labelsize=16)
        ax2.tick_params(axis='y', labelsize=16)
        ax2.set_ylim(3e-6, 3e0)
        ax2.set_yscale('log')
        ax2.grid()
            
        
        handles, labels = ax2.get_legend_handles_labels()
        
        # Figure 
        suptitle = act_str + r' (Layer ' + str(l + 1)
        suptitle += ')' if whitened else ', Unwhitened Data)' #, Subspace ' + str(subspace_type)
        
        fig.suptitle(suptitle, fontsize=28, x=0.6, y=1.125)
        fig.supxlabel(r'Train Epoch', x=0.8, y=-0.06, fontsize=20)
        fig.legend(
            handles,
            labels,
            loc='lower right',
            #bbox_to_anchor=(1.05, 0.82),
            bbox_to_anchor=(1.075, 0.76),
            fontsize=18,
            ncol=3,
            frameon=False
        )
        #fig.supylabel('Subspace Error')

        #plt.tight_layout()
        #fig.subplots_adjust(left=0.08, right=0.98, bottom=0.12, top=0.85, wspace=0.25)
        
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
            
        save_fname = os.path.join(save_dir, act_str + '_network_layer_' + str(l + 1) + '_weight_svd_errors.pdf')
        if savefig:
            plt.savefig(save_fname, bbox_inches='tight', pad_inches=0.02)
        plt.show()
        plt.close()

In [ ]:
end_epoch = epochs



for k, act_str in enumerate(act_fn_strs):
    mean_errors_dict = {
        'U_top_errors': mean_U_top_errors[k, :end_epoch, ...],
        'U_mid_errors': mean_U_mid_errors[k, :end_epoch, ...], 
        'U_bot_errors': mean_U_bot_errors[k, :end_epoch, ...],
        'V_top_errors': mean_V_top_errors[k, :end_epoch, ...],
        'V_mid_errors': mean_V_mid_errors[k, :end_epoch, ...], 
        'V_bot_errors': mean_V_bot_errors[k, :end_epoch, ...],
        'svals': mean_svals[k, ...]
    }

    std_errors_dict = {
        'U_top_errors': std_U_top_errors[k, :end_epoch, ...],
        'U_mid_errors': std_U_mid_errors[k, :end_epoch, ...], 
        'U_bot_errors': std_U_bot_errors[k, :end_epoch, ...],
        'V_top_errors': std_V_top_errors[k, :end_epoch, ...],
        'V_mid_errors': std_V_mid_errors[k, :end_epoch, ...], 
        'V_bot_errors': std_V_bot_errors[k, :end_epoch, ...]
    }
    
    plot_weight_svd_errors(mean_errors_dict, 
                           std_errors_dict, 
                           act_str, 
                           d, 
                           K, 
                           savefig_dir,
                           savefig=save)

# arXiv Teaser Fig

In [ ]:
def plot_series(
    ax,
    series,
    layer=0,
    color="viridis",
    crossing_lines=False,
    x_points=None,
    y_points=None,
    zoom=0.8,
    elev=30,
    azim=-50,
    roll=0,
    alpha=1.0,
    linewidth=1.0,
):
    n_x_indices, n_y_indices = series.shape[0], series.shape[-1]
    x_indices = np.arange(n_x_indices) + 1
    y_indices = np.arange(n_y_indices) + 1

    if x_points is None:
        x_points = x_indices
    if y_points is None:
        y_points = y_indices

    spectrum_verts = []

    for idx in x_indices:
        spectrum_verts.append(
            [
                (0, np.min(series) - 0.1),
                *zip(y_points, series[idx - 1, layer, :]),
                (y_points[-1], np.min(series) - 0.1),
            ]
        )
    
    
    

    stride=5
    
    spectrum_poly = PolyCollection(spectrum_verts[::stride]) #, facecolors='none')
    spectrum_poly.set_linewidth(linewidth)
    spectrum_poly.set_alpha(alpha)
    spectrum_poly.set_facecolor(
        plt.colormaps[color](np.linspace(0.3, 0.8, len(spectrum_verts[::stride])))  # type: ignore
    )
    spectrum_poly.set_edgecolor("black")
    
    
#     colors = plt.colormaps[color](np.linspace(0.5, 1.0 len(spectrum_verts)))
#     spectrum_poly.set_edgecolor(colors)
#     spectrum_poly.set_linewidth(0.8)
#     spectrum_poly.set_alpha(0.9)
    
    #spectrum_poly.set_antialiased(False)

    ax.set_box_aspect(aspect=None, zoom=zoom)

    ax.add_collection3d(spectrum_poly, zs=x_indices[::stride], zdir="y")

    if crossing_lines:
        path_verts = []

        for idx in y_indices:
            path_verts.append([*zip(x_points, series[:, layer, idx - 1])])
        path_line = LineCollection(path_verts)
        path_line.set_linewidth(linewidth)
        path_line.set_edgecolor("black")
        ax.add_collection3d(path_line, zs=y_indices, zdir="x")

    ax.set_xlim(y_points[0], y_points[-1])
    ax.set_ylim(x_points[0], x_points[-1])
    ax.set_zlim(np.min(series) - 0.1, np.max(series) + 0.1)

    ax.view_init(elev, azim, roll)
    ax.xaxis.set_major_locator(MultipleLocator(8)) #MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MultipleLocator(25)) #MaxNLocator(integer=True))
    
    ax.xaxis.labelpad=8
    ax.yaxis.labelpad=8
    

def plot_svals(svals: np.ndarray,
               act_str: str, 
               d: int, 
               K: int, 
               save_dir: str,
               savefig: bool = False, 
               whitened: bool = True):
    

    epochs, L = svals.shape[0], svals.shape[1]
    x = np.arange(epochs) + 1


    for l in range(L):
        fig = plt.figure(figsize=(5, 5), constrained_layout=True)
        
#         gs = fig.add_gridspec(
#             nrows=1,
#             ncols=1,
#             width_ratios=[1.33, 1.0]  # ← give left plot more room
#         )
        
        # Singular values
        ax1 = fig.add_subplot(projection='3d') #fig.add_subplot(131, projection='3d')
        plot_series(ax1, svals, layer=l, crossing_lines=True, linewidth=0.5, alpha=1.0, zoom=1.1)
        
        ax1.set_xlabel(r'SV Index', fontsize=20)
        ax1.set_ylabel(r'Train Epoch', fontsize=20)

        ax1.set_title(act_str, fontsize=20) #+ r' Layer ' + str(l + 1) + ' Singular Values', fontsize=20)
        ax1.xaxis.set_tick_params(labelsize=14)
        ax1.yaxis.set_tick_params(labelsize=14)
        ax1.zaxis.set_tick_params(labelsize=14)
        
        

        
        # Figure 
        #suptitle = act_str + r' (Layer ' + str(l + 1)
        #suptitle += ')' if whitened else ', Unwhitened Data)' #, Subspace ' + str(subspace_type)
        
        #fig.suptitle(suptitle, fontsize=28, x=0.6, y=1.125)
#         fig.supxlabel(r'Train Epoch', x=0.8, y=-0.06, fontsize=20)
#         fig.legend(
#             handles,
#             labels,
#             loc='lower right',
#             #bbox_to_anchor=(1.05, 0.82),
#             bbox_to_anchor=(1.075, 0.76),
#             fontsize=18,
#             ncol=3,
#             frameon=False
#         )
        #fig.supylabel('Subspace Error')

        #plt.tight_layout()
        #fig.subplots_adjust(left=0.08, right=0.98, bottom=0.12, top=0.85, wspace=0.25)
        
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
            
        save_fname = os.path.join(save_dir, act_str + '_network_layer_' + str(l + 1) + '_svals.pdf')
        if savefig:
            plt.savefig(save_fname, bbox_inches='tight', pad_inches=0.02)
        plt.show()
        plt.close()

In [ ]:
end_epoch = epochs

for k, act_str in enumerate(act_fn_strs):
    plot_svals(mean_svals[k, ...], 
               act_str, 
               d,
               K, 
               savefig_dir,
               savefig=save)